# 🚀 Aero-Sense: Advanced ML Pipeline
## Multi-Dataset Predictive Maintenance System

**Google Cloud Partner Hackathon 2025**  
**Datasets:**
- NASA C-MAPSS (FD001-FD004) - 2008
- NASA Turbofan v2 (#17) - 2021 (14.6 GB) - **Most Advanced**
- NASA Batteries (#5) - Aircraft electronics
- NASA Bearings (#4) - Engine components
- FEMTO Bearing (#10) - High-frequency vibration

**Total Data:** ~15+ GB across 5+ datasets

## 📦 Setup & Installations

In [ ]:
# Install required packages
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn
!pip install -q google-cloud-aiplatform google-cloud-storage
!pip install -q plotly kaleido

In [ ]:
# Imports
import os
import requests
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML imports
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Google Cloud imports
from google.cloud import aiplatform, storage

print("✅ All packages imported successfully")
print(f"📅 Notebook started: {datetime.now()}")

## 🌍 Google Cloud Setup

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

# Set project variables
PROJECT_ID = "your-project-id"  # TODO: Update this
REGION = "us-central1"
BUCKET_NAME = f"aero-sense-models-{PROJECT_ID}"

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=REGION)

print(f"✅ Google Cloud configured")
print(f"   Project: {PROJECT_ID}")
print(f"   Region: {REGION}")
print(f"   Bucket: {BUCKET_NAME}")

## 📥 Dataset Download & Extraction

### Strategy:
1. Download all datasets to `/content/data/`
2. Extract and organize by dataset type
3. Keep raw files for reference

In [ ]:
# Create data directory
!mkdir -p /content/data

# Dataset URLs
DATASETS = {
    'turbofan_v1': {
        'url': 'https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip',
        'size': '12 MB',
        'priority': 'HIGH'
    },
    'turbofan_v2': {
        'url': 'https://phm-datasets.s3.amazonaws.com/NASA/17.+Turbofan+Engine+Degradation+Simulation+Data+Set+2.zip',
        'size': '14.6 GB',
        'priority': 'HIGH',
        'note': 'Most advanced dataset (2021)'
    },
    'batteries': {
        'url': 'https://phm-datasets.s3.amazonaws.com/NASA/5.+Battery+Data+Set.zip',
        'size': '~500 MB',
        'priority': 'MEDIUM'
    },
    'bearings': {
        'url': 'https://phm-datasets.s3.amazonaws.com/NASA/4.+Bearings.zip',
        'size': '~300 MB',
        'priority': 'MEDIUM'
    },
    'femto_bearings': {
        'url': 'https://phm-datasets.s3.amazonaws.com/NASA/10.+FEMTO+Bearing.zip',
        'size': '~800 MB',
        'priority': 'LOW'
    }
}

print("📊 Dataset Inventory:")
for name, info in DATASETS.items():
    print(f"  {info['priority']:6} | {info['size']:8} | {name}")
    if 'note' in info:
        print(f"         → {info['note']}")

In [ ]:
def download_dataset(name, url, timeout=7200):
    """
    Download dataset with progress tracking
    """
    output_path = f"/content/data/{name}.zip"
    
    if os.path.exists(output_path):
        print(f"⏭️  {name}: Already downloaded")
        return output_path
    
    print(f"⬇️  Downloading {name}...")
    start_time = datetime.now()
    
    try:
        # Stream download with progress
        response = requests.get(url, stream=True, timeout=timeout)
        total_size = int(response.headers.get('content-length', 0))
        
        with open(output_path, 'wb') as f:
            downloaded = 0
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size > 0:
                        progress = (downloaded / total_size) * 100
                        if downloaded % (50 * 1024 * 1024) == 0:  # Log every 50MB
                            print(f"   Progress: {progress:.1f}% ({downloaded/(1024**2):.0f} MB / {total_size/(1024**2):.0f} MB)")
        
        elapsed = (datetime.now() - start_time).total_seconds()
        speed = (downloaded / (1024**2)) / elapsed if elapsed > 0 else 0
        print(f"✅ {name}: Downloaded in {elapsed:.0f}s ({speed:.1f} MB/s)")
        return output_path
        
    except Exception as e:
        print(f"❌ {name}: Error - {e}")
        return None

def extract_dataset(zip_path, extract_to=None):
    """
    Extract ZIP file
    """
    if extract_to is None:
        extract_to = os.path.dirname(zip_path)
    
    dataset_name = os.path.basename(zip_path).replace('.zip', '')
    extract_path = os.path.join(extract_to, dataset_name)
    
    if os.path.exists(extract_path):
        print(f"⏭️  {dataset_name}: Already extracted")
        return extract_path
    
    print(f"📦 Extracting {dataset_name}...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"✅ {dataset_name}: Extracted")
        return extract_path
    except Exception as e:
        print(f"❌ {dataset_name}: Extraction error - {e}")
        return None

### Download HIGH Priority Datasets First

In [ ]:
# Download Turbofan v1 (small, quick)
print("="*60)
print("HIGH PRIORITY: Turbofan v1 (C-MAPSS)")
print("="*60)
zip_path = download_dataset('turbofan_v1', DATASETS['turbofan_v1']['url'])
if zip_path:
    extract_dataset(zip_path)

In [ ]:
# Download Turbofan v2 (large, but most important)
print("="*60)
print("HIGH PRIORITY: Turbofan v2 (2021 - Most Advanced)")
print("⚠️  This is 14.6 GB and will take 10-30 minutes")
print("="*60)
zip_path = download_dataset('turbofan_v2', DATASETS['turbofan_v2']['url'], timeout=10800)
if zip_path:
    extract_dataset(zip_path)

### Download MEDIUM Priority Datasets

In [ ]:
# Download Batteries
print("="*60)
print("MEDIUM PRIORITY: Battery Data")
print("="*60)
zip_path = download_dataset('batteries', DATASETS['batteries']['url'])
if zip_path:
    extract_dataset(zip_path)

In [ ]:
# Download Bearings
print("="*60)
print("MEDIUM PRIORITY: Bearing Data")
print("="*60)
zip_path = download_dataset('bearings', DATASETS['bearings']['url'])
if zip_path:
    extract_dataset(zip_path)

In [ ]:
# Download FEMTO Bearings (optional)
print("="*60)
print("LOW PRIORITY: FEMTO Bearing Data")
print("="*60)
zip_path = download_dataset('femto_bearings', DATASETS['femto_bearings']['url'])
if zip_path:
    extract_dataset(zip_path)

## 📊 Data Exploration & Analysis

In [ ]:
# List downloaded files
!echo "📂 Downloaded datasets:"
!ls -lh /content/data/*.zip 2>/dev/null | awk '{print $5, $9}'
!echo ""
!echo "📁 Extracted directories:"
!ls -d /content/data/*/ 2>/dev/null

### Turbofan v1 (C-MAPSS) Analysis

In [ ]:
# Column names for C-MAPSS
CMAPSS_COLUMNS = ['unit_id', 'cycle', 'setting1', 'setting2', 'setting3'] + \
                 [f'sensor{i}' for i in range(1, 22)]

def load_cmapss_dataset(dataset_name='FD001'):
    """
    Load and preprocess C-MAPSS dataset
    """
    base_path = '/content/data/turbofan_v1'
    
    # Find the actual path (might be nested)
    for root, dirs, files in os.walk(base_path):
        train_file = f'train_{dataset_name}.txt'
        if train_file in files:
            train_path = os.path.join(root, train_file)
            test_path = os.path.join(root, f'test_{dataset_name}.txt')
            rul_path = os.path.join(root, f'RUL_{dataset_name}.txt')
            
            # Load data
            train = pd.read_csv(train_path, sep=r'\s+', header=None, names=CMAPSS_COLUMNS)
            test = pd.read_csv(test_path, sep=r'\s+', header=None, names=CMAPSS_COLUMNS)
            rul = pd.read_csv(rul_path, header=None, names=['RUL'])
            
            return train, test, rul
    
    raise FileNotFoundError(f"Dataset {dataset_name} not found")

# Load FD001
try:
    train_fd001, test_fd001, rul_fd001 = load_cmapss_dataset('FD001')
    print("✅ FD001 loaded successfully")
    print(f"   Train: {len(train_fd001):,} samples, {train_fd001['unit_id'].nunique()} engines")
    print(f"   Test:  {len(test_fd001):,} samples, {test_fd001['unit_id'].nunique()} engines")
    print(f"\n📊 First few rows:")
    display(train_fd001.head())
except Exception as e:
    print(f"❌ Error loading FD001: {e}")

## 🤖 Model Training Pipeline

### Multi-Dataset XGBoost Strategy

In [ ]:
# Model training will continue in next cells...
print("🚀 Ready for model training!")
print("\n📋 Next Steps:")
print("  1. Feature engineering")
print("  2. RUL calculation")
print("  3. XGBoost training (FD001, FD002, FD003, FD004)")
print("  4. Turbofan v2 model training")
print("  5. Model export to Vertex AI")

---
## 💾 Save Progress

Notebook will continue with:
- Feature engineering
- RUL computation
- Multi-dataset training
- Model evaluation
- Vertex AI deployment